# Denim Atlas — frozen calibration comparison

Private Kaggle GPU run. Upload only `denim-calibration-input.zip` as a private input, enable Internet and GPU. Contains licensed crop bytes and private rater judgments; do not publish notebook inputs/outputs. No training or held-out research claim. The bundle pins crop review, source notices, labels and the scoring source used for this run.

Outputs: frozen features, calibration-only B0/B1 report (w=0.5, fixed before labels), provenance hashes. Q1 is incomplete because its conflicting repeat is excluded, so no nDCG is emitted for it. Q2/Q3 can have descriptive calibration nDCG only. Save a version **with output files**, then stop the session.


In [ ]:
from pathlib import Path
import subprocess, sys, hashlib, json, zipfile
import torch
assert Path('/kaggle/working').is_dir(), 'Remote Kaggle only'
assert torch.cuda.is_available(), 'Enable GPU first; no CPU fallback'
print('GPU:', torch.cuda.get_device_name(0), flush=True)
subprocess.run([sys.executable,'-m','pip','install','-q','open_clip_torch==3.3.0'],check=True)


In [ ]:
import shutil
expected_files={'crops/00.png': '5b4afef60ebbf220c50794bf31d9573a43bba8994c78fa58a8aa81c941f19ea8', 'crops/01.png': '04eb48b625f1a7fe3770f59e4fa2312a3742fb6bdaecde9c087f106dce6bfec7', 'crops/02.png': 'c95957bb3de4d7e4be227b4dcb161b9de2518a43c4d499be8f05659db5a59090', 'crops/03.png': '336c0f773bf52dfb562c5e53e6edd30234b5c3989f8ffe9ff7ceb92e7a5b445c', 'crops/04.png': '6d927f02f79cc40b9a6620598e1380494b22136934f18f0e722ff6e1821cebc7', 'crops/05.png': 'd3cd03c14dda9cc1aa7dafe765795d702dd0348cc6374ab2ead18155f16191b6', 'crops/06.png': '8c6857ac54e5b2ae08c09724c91b5bc6fe2e71e2e8c56fcdb7f7f87e750351ac', 'crops/07.png': '9f56dd566f3bb582daf92141dd62f864973e50566e6e146ce6ee1d86ad4c221f', 'crops/08.png': '503a7a4fe54ecb2375eee819fc76f599cf38fdead05604ca5f0b834e0134228b', 'crops/09.png': '165a5ab6c85910847785d6b39737cec4d954f5c072f20d36eb7f9075c36f4b31', 'crops/10.png': 'c73503a8d370c3d0cc2ccc508d3e37256db7d292e8513e9a55c240e8fa9f2624', 'crops/11.png': 'cdec9210d9d77e2c811270e434b021ec480292677bab6817be93dad1e9a5102c', 'crops/12.png': 'e2a236e611c002ff3ff1645f797919eb8cbda610382e8139894c1e7a0dfd616b', 'crops/13.png': 'fff3838f80a5e8a97803aad19e808989c253d6e7cc119a158636290562940bc4', 'crops/14.png': 'c4bc84348f607b04d0a673972bb8c968fc0f23ddb2dcc605ae7c404875153e2d', 'crops/15.png': '4ce729a35e522ffda85946b50d39ad5e8df4abeb9819feadc98680c5056209e7', 'crops/16.png': '28cd9f9efced5f07a2a6fee3404fc47a0b91950680b095b41e113b90c8681769', 'crops/17.png': '1f369a224407193bc8f2d6c51aef0af764ff1305deddcbf29bf054dc60413456', 'crops/18.png': '48f6549ba285ec0ac513164316cea70a7ab472b5fe77a2a10eddbf6bf70dea1d', 'crops/19.png': '35903ccd8eea215ee36dfa5e2cdb50f503eb829f58d4cf8732156887e75735cb', 'crops/20.png': '4f522355a02f62d996e24502441484a087275400033ece7a0aaae91194f7d40b', 'crops/21.png': '468fc42e6831b5c5cfe322f9f8808220f9f559b7b94a601eed1227296e3cd87e', 'crops/22.png': '9efa9c54d2193b68165e1ca3b56609094a61a0a265338198fd8f6a7260270d90', 'crops/23.png': 'db36669ff69791ffb3b453ce7629b282fb93ecb8d9325284f4a688b91347bce4', 'crops/24.png': '73fc28d476fcc4d88c5d3b97d28faf8a6ecb4350f932f4a05078dbbc95c94fbb', 'crops/25.png': '15abdc4bbe2626a34e2da353877b4fd8ec5761de0552c75c97901924cfe63e91', 'crops/26.png': 'bdfbc3b1864485cf0d25bd61f86f521a67197a3a01468f98ad5c3e437f0944ff', 'manifest.json': 'a1a07b0be05684ee72104757069fbaed201f229f4786dc3c112d88ee98bc4d88', 'packet.json': 'c5fe3f98f79a9b402f64cbfc8f76dd3477bde5d74c7fb980046c77ca6edea07a', 'ratings.json': '6011b702c9fc146e247e90f1a9f12f15f6753e294991dc890f74fd90d87d878f', 'scripts/analyze_calibration.py': '735610bff5c29438e5534e86e259c668cb3d54cdaf102ba9506ab5ceefbf5ffe', 'scripts/evaluate.py': '9454d6bc98bb736ab002d65c4ff0ace9c92a64cb63ff1916e3c6efa7f0aee9d3', 'scripts/extract_features.py': 'b42ec474d6cb378cc94f3f250e5f999b83c259903f2bba9db6839e4ffabe1d68', 'scripts/import_judgments.py': 'a66b03b3041d97811231133f4c94a25a0cfc77733a1b44c0cb7322ea877667a5', 'scripts/run_baselines.py': '426784222e66c026c7b044cf23920e54a1b1889dbe010255b1298f27721dfa01'}
manifests=list(Path('/kaggle/input').rglob('manifest.json'))
assert len(manifests)==1, 'Attach exactly one private calibration input'
source=manifests[0].parent
actual={p.relative_to(source).as_posix() for p in source.rglob('*') if p.is_file()}
assert actual==set(expected_files), 'Unexpected extracted input members'
for name,digest in expected_files.items():
    assert hashlib.sha256((source/name).read_bytes()).hexdigest()==digest, 'Changed input: '+name
root=Path('/kaggle/working/calibration_round1_v1')
shutil.copytree(source,root)
subprocess.run([sys.executable,str(root/'scripts/extract_features.py'),
 '--manifest',str(root/'manifest.json'),'--image-root',str(root),
 '--output',str(root/'features.json')],check=True)
subprocess.run([sys.executable,str(root/'scripts/analyze_calibration.py'),
 '--packet',str(root/'packet.json'),'--ratings',str(root/'ratings.json'),
 '--features',str(root/'features.json'),'--output',str(root/'comparison.json')],check=True)
result=json.loads((root/'comparison.json').read_text())
assert result['purpose']=='calibration_only' and not result['held_out_evaluation']
print(json.dumps({'usable_pairs':result['usable_pairs'],'results':[
 {k:r[k] for k in ('baseline','query_id','fully_judged_pool','calibration_ndcg_at_5','top5')}
 for r in result['results']]},indent=2),flush=True)
